# AutoMode NeurIPS — Figure Reproduction

This notebook reads the logged JSONs from the three driver runs and produces the six figures the paper needs:

1. **Results bar chart** — accuracy × method × model (the headline)
2. **Accuracy-vs-trainable-params Pareto** — shows AutoMode dominates the frontier
3. **Training loss curves** — proves all methods trained stably
4. **Trainable-parameter trajectory** — shows AutoMode's dynamic capacity use
5. **Per-layer FFT selection heatmap** — which layers AutoMode picks (over training)
6. **Layer-mode timeline** — alternative view of figure 5, per-seed
7. **LR schedule sanity** — proves the sawtooth bug is fixed (monotone decay)

All figures use matplotlib (NeurIPS style — no color-only distinctions).

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'legend.fontsize': 9,
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

FIG_DIR = Path('paper/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load all results from the three GPU CSVs
dfs = []
for csv in ['runs/gpu0_results.csv', 'runs/gpu1_results.csv', 'runs/gpu2_results.csv']:
    if Path(csv).exists():
        dfs.append(pd.read_csv(csv))
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f'{len(df)} total completed runs')
print(df['status'].value_counts() if 'status' in df.columns else 'no status col')
# ── Split by paper_fidelity for honest headline vs supplementary reporting ──
# v0.2.0+ writes a paper_fidelity column to every result row. Headline
# figures should include only faithful baselines; approximate ones go in
# a supplementary section. If the column is missing (pre-v0.2.0 CSVs),
# fall back to including everything with a warning.
if 'paper_fidelity' in df.columns:
    df_faithful = df[df['paper_fidelity'] == 'faithful'].copy()
    df_approx = df[df['paper_fidelity'] == 'approximate'].copy()
    print(f'  faithful baselines: {len(df_faithful)} rows '
          f'({sorted(df_faithful.method.unique())})')
    print(f'  approximate baselines: {len(df_approx)} rows '
          f'({sorted(df_approx.method.unique()) if len(df_approx) else []})')
else:
    print('  WARNING: no paper_fidelity column; results predate v0.2.0. '
          'Cannot split headline vs supplementary automatically.')
    df_faithful = df
    df_approx = df.iloc[0:0]


In [ ]:
# ── Figure 1: Results bar chart ───────────────────────────────────────────
# For each (model, track), plot mean ± std eval accuracy across methods
# df_ok kept for back-compat; headline plots below use df_faithful_ok
df_ok = df[df['status'] == 'ok'].copy()
df_faithful_ok = df_faithful[df_faithful['status'] == 'ok'].copy() if 'paper_fidelity' in df.columns else df_ok
df_approx_ok = df_approx[df_approx['status'] == 'ok'].copy() if 'paper_fidelity' in df.columns else df_ok.iloc[0:0]
if 'eval_gsm8k_maj1' not in df_ok.columns:
    df_ok['eval_gsm8k_maj1'] = None

def plot_bar(df_sub, metric, title, save_path):
    g = df_sub.groupby(['model_name', 'method'])[metric].agg(['mean', 'std', 'count']).reset_index()
    models = g['model_name'].unique()
    methods = sorted(g['method'].unique())
    fig, ax = plt.subplots(figsize=(10, 4.5))
    x = np.arange(len(models))
    w = 0.8 / len(methods)
    hatches = ['', '/', '\\\\', 'xx', '..', '++', 'oo', '--']
    for i, method in enumerate(methods):
        means = [g[(g.model_name == m) & (g.method == method)]['mean'].values[0] if not g[(g.model_name == m) & (g.method == method)].empty else 0 for m in models]
        stds = [g[(g.model_name == m) & (g.method == method)]['std'].values[0] if not g[(g.model_name == m) & (g.method == method)].empty else 0 for m in models]
        ax.bar(x + i * w, means, w, yerr=stds, label=method,
               hatch=hatches[i % len(hatches)], edgecolor='black', linewidth=0.6)
    ax.set_xticks(x + w * (len(methods) - 1) / 2)
    ax.set_xticklabels([m.split('/')[-1] for m in models], rotation=15, ha='right')
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.legend(loc='upper right', ncol=2, fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.show()

if not df_ok.empty and 'eval_gsm8k_maj1' in df_ok.columns:
    plot_bar(
        df_faithful_ok.dropna(subset=['eval_gsm8k_maj1']),
        'eval_gsm8k_maj1',
        'GSM8K maj@1 by method (mean ± std across seeds)',
        FIG_DIR / 'fig1_results_bar_gsm8k.pdf',
    )

In [ ]:
# ── Figure 2: Accuracy vs Trainable params Pareto ─────────────────────────
fig, ax = plt.subplots(figsize=(6, 4.5))
method_markers = {'full_ft': 's', 'lora': 'o', 'automode': '*',
                   'topk_deep_block': '^', 'lisa': 'v', 'adalora': 'D',
                   'adagradselect': 'P', 'dora': 'X'}
for method, marker in method_markers.items():
    sub = df_faithful_ok[df_faithful_ok['method'] == method]
    if sub.empty or 'eval_gsm8k_maj1' not in sub.columns:
        continue
    x = sub['final_trainable_params'] / 1e6   # M params
    y = sub['eval_gsm8k_maj1']
    size = 80 if method == 'automode' else 50
    ax.scatter(x, y, marker=marker, s=size, label=method,
               alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xscale('log')
ax.set_xlabel('Trainable parameters (M, log scale)')
ax.set_ylabel('GSM8K maj@1')
ax.set_title('Accuracy vs compute — Pareto frontier')
ax.legend(loc='lower right', fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_pareto.pdf')
plt.show()

In [ ]:
# ── Figure 3: Training loss curves (one panel per method, all seeds overlaid) ─
def load_per_run_log(run_id, log_name, root='runs'):
    # Search all tier subdirs
    for tier_dir in Path(root).rglob(run_id):
        path = tier_dir / 'logs' / log_name
        if path.exists():
            return json.load(open(path))
    return None

# Pick one model for the representative figure
focal_model = 'google/gemma-2-2b'
focal_track = 'metamath'
df_focal = df_ok[(df_ok['model_name'] == focal_model) & (df_ok['train_track'] == focal_track)]

methods_to_plot = ['full_ft', 'lora', 'automode']
fig, axes = plt.subplots(1, len(methods_to_plot), figsize=(13, 3.2), sharey=True)
for ax, method in zip(axes, methods_to_plot):
    sub = df_focal[df_focal['method'] == method]
    for _, row in sub.iterrows():
        log = load_per_run_log(row['run_id'], 'training_loss.json')
        if not log:
            continue
        steps = [e['step'] for e in log['loss_log']]
        losses = [e['loss'] for e in log['loss_log']]
        ax.plot(steps, losses, alpha=0.7, label=f'seed {row["seed"]}')
    ax.set_title(method)
    ax.set_xlabel('opt step')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('training loss')
fig.suptitle(f'{focal_model.split("/")[-1]}, {focal_track}', y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_loss_curves.pdf')
plt.show()

In [ ]:
# ── Figure 4: Trainable-parameter trajectory (AutoMode dynamic capacity) ──
fig, ax = plt.subplots(figsize=(8, 3.5))
automode_row = df_focal[df_focal['method'] == 'automode'].iloc[0]
lora_row = df_focal[df_focal['method'] == 'lora'].iloc[0]
full_row = df_focal[df_focal['method'] == 'full_ft'].iloc[0]

for label, row, color in [
    ('Full-FT', full_row, 'black'),
    ('LoRA r=16', lora_row, 'tab:blue'),
    ('AutoMode', automode_row, 'tab:red'),
]:
    log = load_per_run_log(row['run_id'], 'trainable_params.json')
    if log:
        steps = [e['step'] for e in log['param_log']]
        counts = [e['trainable_params'] / 1e6 for e in log['param_log']]
        ax.plot(steps, counts, label=label, color=color, linewidth=1.5)

ax.set_xlabel('optimizer step')
ax.set_ylabel('trainable parameters (M)')
ax.set_yscale('log')
ax.set_title('Dynamic capacity — AutoMode adapts during training')
ax.legend(loc='center right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_trainable_params.pdf')
plt.show()

In [ ]:
# ── Figure 5: Per-layer FFT selection frequency heatmap ───────────────────
# Which layers does AutoMode choose to upgrade to FFT, across all (model, seed) runs?
automode_runs = df_ok[df_ok['method'] == 'automode']

def load_timeline(run_id, root='runs'):
    for d in Path(root).rglob(run_id):
        p = d / 'dynamic' / 'layer_timeline.json'
        if p.exists():
            return json.load(open(p))
    return None

# Aggregate by model (different layer counts per model)
for model in automode_runs['model_name'].unique():
    sub = automode_runs[automode_runs['model_name'] == model]
    timelines = []
    for _, r in sub.iterrows():
        tl = load_timeline(r['run_id'])
        if tl:
            timelines.append(tl['layer_timeline'])
    if not timelines:
        continue
    # Build matrix: rows = snapshots, cols = layer_id (sorted numerically)
    all_layers = sorted({k for tl in timelines for snap in tl for k in snap['modes']},
                         key=lambda s: int(s.split('_')[1]))
    # Fraction of snapshots where each layer was in full_ft mode
    frac_fft = {l: 0 for l in all_layers}
    tot_snaps = 0
    for tl in timelines:
        for snap in tl:
            tot_snaps += 1
            for l, m in snap['modes'].items():
                if m == 'full_ft':
                    frac_fft[l] += 1
    fracs = [frac_fft[l] / max(tot_snaps, 1) for l in all_layers]
    fig, ax = plt.subplots(figsize=(9, 1.8))
    im = ax.imshow([fracs], aspect='auto', cmap='RdYlBu_r', vmin=0, vmax=1)
    ax.set_xticks(range(len(all_layers)))
    ax.set_xticklabels([l.replace('layer_', '') for l in all_layers], fontsize=7)
    ax.set_yticks([])
    ax.set_xlabel('transformer layer index')
    ax.set_title(f'AutoMode FFT selection frequency — {model.split("/")[-1]}')
    plt.colorbar(im, ax=ax, label='frac. of snapshots in FFT mode', fraction=0.03)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'fig5_layer_freq_{model.split("/")[-1]}.pdf')
    plt.show()

In [ ]:
# ── Figure 7: LR schedule sanity (proves no sawtooth) ─────────────────────
# This is the most important figure for reviewer trust — any reviewer who
# knows the prior sawtooth bug exists will look for this.
fig, ax = plt.subplots(figsize=(8, 3.5))
for label, row, color, ls in [
    ('LoRA (no switching)', lora_row, 'tab:blue', '-'),
    ('AutoMode (with switching)', automode_row, 'tab:red', '-'),
]:
    log = load_per_run_log(row['run_id'], 'lr_schedule.json')
    if log:
        steps = [e['step'] for e in log['lr_log']]
        lrs = [e['lr'] for e in log['lr_log']]
        ax.plot(steps, lrs, label=label, color=color, linestyle=ls, linewidth=1.3)
ax.set_xlabel('optimizer step')
ax.set_ylabel('learning rate')
ax.set_title('LR schedule: monotonic cosine decay preserved across switches')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_lr_schedule.pdf')
plt.show()

In [ ]:
# ── Tables ────────────────────────────────────────────────────────────────
# Produce the main results table for LaTeX
metric_cols = [c for c in df_ok.columns if c.startswith('eval_') and ('maj1' in c or '_acc' in c)]
if metric_cols:
    table = (df_faithful_ok.groupby(['model_name', 'method', 'train_track'])[metric_cols]
                   .agg(['mean', 'std'])
                   .round(4))
    print(table.to_string())
    # Save as CSV for paste into LaTeX via csvsimple or similar
    table.to_csv('paper/main_results_table.csv')
    print(f'\nWritten to paper/main_results_table.csv')

In [ ]:
# ── Statistical tests ─────────────────────────────────────────────────────
# For each (model, track), test automode vs lora and automode vs full_ft
# with paired-t or Wilcoxon, Holm-Bonferroni corrected.
from scipy import stats

def pairwise_significance(df_ok, metric):
    rows = []
    for (model, track), sub in df_ok.groupby(['model_name', 'train_track']):
        methods = sub['method'].unique()
        if 'automode' not in methods:
            continue
        am = sub[sub['method'] == 'automode'][metric].values
        if len(am) < 2:
            continue
        for other in ['full_ft', 'lora']:
            if other not in methods:
                continue
            ot = sub[sub['method'] == other][metric].values
            if len(ot) < 2:
                continue
            n = min(len(am), len(ot))
            t_stat, p_val = stats.ttest_rel(am[:n], ot[:n])
            rows.append({
                'model': model.split('/')[-1],
                'track': track,
                'comparison': f'automode vs {other}',
                'delta_mean': float(np.mean(am[:n]) - np.mean(ot[:n])),
                'p_value': float(p_val),
                'n_seeds': int(n),
            })
    return pd.DataFrame(rows)

if 'eval_gsm8k_maj1' in df_ok.columns:
    sig = pairwise_significance(df_faithful_ok.dropna(subset=['eval_gsm8k_maj1']),
                                 'eval_gsm8k_maj1')
    # Holm-Bonferroni
    sig = sig.sort_values('p_value').reset_index(drop=True)
    m = len(sig)
    sig['p_holm'] = [min(1.0, p * (m - i)) for i, p in enumerate(sig['p_value'])]
    sig['p_holm'] = sig['p_holm'].cummax()  # enforce monotonic
    print(sig.round(4).to_string())
    sig.to_csv('paper/significance_tests.csv', index=False)

In [ ]:
# ── Supplementary: approximate baselines (LISA, AdaGradSelect, LoRA-GA) ──
# These are kept in a separate section per paper_fidelity = 'approximate'.
# They should NOT appear in the main results table or the headline bar chart;
# instead, include them in an appendix or supplementary table with an
# explicit 'implementation approximate' note.
if not df_approx_ok.empty:
    print('Approximate (supplementary) baselines:')
    metric_cols = [c for c in df_approx_ok.columns 
                    if c.startswith('eval_') and ('maj1' in c or '_acc' in c)]
    if metric_cols:
        supp_table = (df_approx_ok.groupby(['model_name', 'method', 'train_track'])[metric_cols]
                                   .agg(['mean', 'std', 'count'])
                                   .round(4))
        print(supp_table.to_string())
        supp_table.to_csv('paper/supplementary_approximate_baselines.csv')
        print('\nWritten to paper/supplementary_approximate_baselines.csv')
else:
    print('No approximate baselines in results (Tier 4 not run or not yet complete).')